# Importação de bibliotecas

In [2]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import average_precision_score
from sklearn.preprocessing import LabelBinarizer

# Definição de caminhos

In [3]:
# --- Caminhos para os arquivos ---
TRAIN_FILE_PATH = '../data/extracted_features/train_features_completas.csv'
TEST_FILE_PATH = '../data/extracted_features/test_features_completas.csv'

# --- Carregar os dados ---
try:
    train_df = pd.read_csv(TRAIN_FILE_PATH)
    test_df = pd.read_csv(TEST_FILE_PATH)
    print(f"Arquivos carregados com sucesso!")
    print(f"Formato dos dados de treino: {train_df.shape}")
    print(f"Formato dos dados de teste:  {test_df.shape}")
except FileNotFoundError:
    print(f"Erro: Arquivos não encontrados.")
    print(f"Verifique se os caminhos '{TRAIN_FILE_PATH}' e '{TEST_FILE_PATH}' estão corretos.")
    print("Se estiver no Google Colab, certifique-se de que os arquivos foram enviados e o caminho está correto.")

train_df = train_df.dropna(subset=['Classe'])
test_df = test_df.dropna(subset=['Classe'])


print("\n--- Amostra dos Dados de Treino ---")
display(train_df.head())

/tmp/ipykernel_16288/573660308.py:7: DtypeWarning: Columns (1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv(TRAIN_FILE_PATH)


Arquivos carregados com sucesso!
Formato dos dados de treino: (367411, 68)
Formato dos dados de teste:  (99731, 68)

--- Amostra dos Dados de Treino ---


,nameseq,AAA,AAC,AAG,AAT,ACA,ACC,ACG,ACT,AGA,...,TGC,TGG,TGT,TTA,TTC,TTG,TTT,label,Sequência de TE,Classe
0,ID_0_3_LTR,0.025262,0.01505,0.026337,0.019215,0.018678,0.011825,0.007659,0.01384,0.02365,...,0.026068,0.01935,0.020156,0.015318,0.016931,0.031846,0.026337,ALL_CLASSES,GGACCAGCACAGTACCACCGTTGAACGATTATTCAGCTTGCTTTTG...,LTR
1,ID_1_5_LINE,0.022866,0.015244,0.019817,0.019817,0.021341,0.01372,0.006098,0.019817,0.015244,...,0.007622,0.016768,0.012195,0.015244,0.027439,0.02439,0.018293,ALL_CLASSES,AAACTTCTTGCGCTCATGGCGCTTAGAAGTGACGGACAGCGTGTCG...,LINE
2,ID_2_3_TIR,0.029851,0.014925,0.037313,0.022388,0.029851,0.029851,0.007463,0.007463,0.007463,...,0.007463,0.022388,0.0,0.007463,0.007463,0.007463,0.014925,ALL_CLASSES,ACATTCTAAACAAGTTTGGAATGAAGGATGCCAAGCCCATCAAGAC...,TIR
3,ID_3_10_MITE,0.014493,0.014493,0.021739,0.014493,0.007246,0.0,0.0,0.036232,0.007246,...,0.014493,0.043478,0.014493,0.0,0.028986,0.065217,0.021739,ALL_CLASSES,TGTCTTGATGGGCTTGGCATCCTTCATTCCAAACTTGGTAAGTATA...,MITE
4,ID_4_5_TIR,0.017321,0.011135,0.009898,0.018868,0.008042,0.015775,0.007733,0.017012,0.011135,...,0.016703,0.016084,0.020724,0.025363,0.015156,0.019487,0.030003,ALL_CLASSES,GCAAGCCCTGGTTTTATGCATAACCCTTATATATATGCTATTTTAC...,TIR


# Preparação dos dados

In [4]:
# 1. Separar features (X) e labels (y)
# Primeiro, separamos os labels
y_train = train_df['Classe']
y_test = test_df['Classe']

# Em seguida, selecionamos apenas as colunas de features (k-mers)
# Adicione 'Classe' à lista de colunas a serem descartadas
feature_names = train_df.drop(columns=['nameseq', 'label', 'Sequência de TE', 'Classe']).columns
X_train = train_df[feature_names]
X_test = test_df[feature_names]

print(f"Número de features: {len(feature_names)}")
print(f"Exemplo de features: {feature_names[:5].to_list()}...")

# 2. FORÇAR A CONVERSÃO DE X PARA NUMÉRICO (A CORREÇÃO)
# O erro 'ValueError' indica que há strings ('AAA') nas colunas de features.
# 'errors='coerce'' transformará essas strings problemáticas em 'NaN' (Not a Number).
X_train = X_train.apply(pd.to_numeric, errors='coerce')
X_test = X_test.apply(pd.to_numeric, errors='coerce')

# 3. Verificar se a coerção criou valores NaN (o que indica dados "sujos")
nan_in_train = X_train.isna().sum().sum()
if nan_in_train > 0:
    print(f"\nAlerta: {nan_in_train} valores não numéricos (strings) foram encontrados")
    print("nas colunas de features do treino e convertidos para NaN.")
    
    # Estratégia de Pré-processamento: Preencher NaNs com a média da coluna.
    # Isso permite que o StandardScaler funcione.
    X_train = X_train.fillna(X_train.mean())
    print("Valores NaN foram preenchidos com a média da sua respectiva coluna.")
else:
    print("\nColunas de features do treino parecem ser 100% numéricas.")

# 4. Aplicar o mesmo para o conjunto de teste
nan_in_test = X_test.isna().sum().sum()
if nan_in_test > 0:
    print(f"Alerta: {nan_in_test} valores não numéricos encontrados no teste.")
    # IMPORTANTE: Preencher NaNs do teste com a média do TREINO
    X_test = X_test.fillna(X_train.mean()) 
    print("Valores NaN do teste foram preenchidos com a média do TREINO.")

# 5. Aplicar LabelEncoder nos labels (y)
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

print(f"\nClasses originais: {le.classes_}")
print(f"Classes codificadas: {np.unique(y_train_encoded)}")

Número de features: 64
Exemplo de features: ['AAA', 'AAC', 'AAG', 'AAT', 'ACA']...

Colunas de features do treino parecem ser 100% numéricas.

Classes originais: ['Helitron' 'LINE' 'LTR' 'MITE' 'SINE' 'TIR']
Classes codificadas: [0 1 2 3 4 5]


In [5]:

# 3. Aplicar StandardScaler nas features (X)
scaler = StandardScaler()

# Ajustar o scaler com os dados de TREINO
X_train_scaled = scaler.fit_transform(X_train)

# Apenas transformar os dados de TESTE (usando o ajuste do treino)
X_test_scaled = scaler.transform(X_test)

print(f"Shape dos dados de treino escalados: {X_train_scaled.shape}")

Shape dos dados de treino escalados: (367410, 64)


# Validação cruzada para seleção de hiperparâmetros

In [6]:
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier

param_dist_xgb = {
    'n_estimators': [100, 200],
    'max_depth': [3, 6],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

xgb_base = XGBClassifier(
    random_state=42,
    tree_method='hist',   # novo padrão
    device='cuda',        # força uso da GPU
    eval_metric='logloss'
)

random_search_xgb = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist_xgb,
    n_iter=10,
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=1,   # CRÍTICO para não travar WSL
    scoring='accuracy'
)

random_search_xgb.fit(X_train, y_train_encoded)
print(f"Melhores hiperparâmetros XGBoost: {random_search_xgb.best_params_}")


Fitting 3 folds for each of 10 candidates, totalling 30 fits


/home/gabyl/projetos/trabalho-TEsClassification/venv/lib/python3.12/site-packages/xgboost/core.py:729: UserWarning: [12:36:02] WARNING: /workspace/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


[CV] END colsample_bytree=1.0, learning_rate=0.1, max_depth=6, n_estimators=100, subsample=1.0; total time=  18.2s
[CV] END colsample_bytree=1.0, learning_rate=0.1, max_depth=6, n_estimators=100, subsample=1.0; total time=  10.4s
[CV] END colsample_bytree=1.0, learning_rate=0.1, max_depth=6, n_estimators=100, subsample=1.0; total time=   0.8s
[CV] END colsample_bytree=0.8, learning_rate=0.1, max_depth=6, n_estimators=200, subsample=1.0; total time=  16.9s
[CV] END colsample_bytree=0.8, learning_rate=0.1, max_depth=6, n_estimators=200, subsample=1.0; total time=  16.3s
[CV] END colsample_bytree=0.8, learning_rate=0.1, max_depth=6, n_estimators=200, subsample=1.0; total time=  17.1s
[CV] END colsample_bytree=1.0, learning_rate=0.1, max_depth=3, n_estimators=100, subsample=0.8; total time=   4.9s
[CV] END colsample_bytree=1.0, learning_rate=0.1, max_depth=3, n_estimators=100, subsample=0.8; total time=   4.9s
[CV] END colsample_bytree=1.0, learning_rate=0.1, max_depth=3, n_estimators=100,

In [8]:
# Importar as bibliotecas necessárias
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils.class_weight import compute_sample_weight
from imblearn.over_sampling import SMOTE
import numpy as np

# --- Loop para testar todas as estratégias de balanceamento ---
strategies = ['sample_weight', 'smote', 'none']

# Imprimir os melhores parâmetros uma única vez
best_params_from_search = random_search_xgb.best_params_
print(f"Usando os melhores hiperparâmetros da busca em todos os testes: {best_params_from_search}")

for strategy in strategies:
    print(f"\n{'='*25} INICIANDO TREINAMENTO COM A ESTRATÉGIA: '{strategy.upper()}' {'='*25}\n")

    X_train_final = X_train_scaled
    y_train_final = y_train_encoded
    fit_params = {}

    if strategy == 'sample_weight':
        print("Estratégia de balanceamento: Usando 'sample_weight'.")
        sample_weights = compute_sample_weight(class_weight='balanced', y=y_train_encoded)
        fit_params['sample_weight'] = sample_weights

    elif strategy == 'smote':
        print("Estratégia de balanceamento: Usando SMOTE. Isso pode levar alguns minutos...")
        smote = SMOTE(random_state=42
        )
        X_train_final, y_train_final = smote.fit_resample(X_train_scaled, y_train_encoded)
        print("SMOTE concluído.")
        print(f"  Shape original dos dados de treino: {X_train_scaled.shape}")
        print(f"  Shape dos dados de treino após SMOTE: {X_train_final.shape}")

    else:
        print("Nenhuma estratégia de balanceamento selecionada (treinamento padrão).")

    # 1. Instanciar o classificador
    final_xgb_model = xgb.XGBClassifier(
        **best_params_from_search,
        random_state=42,
        tree_method='hist',
        device='cuda',
        eval_metric='mlogloss',
        use_label_encoder=False
    )

    # 2. Treinar o modelo final com os dados e a estratégia escolhida
    print("\nIniciando o treinamento do modelo final...")
    final_xgb_model.fit(X_train_final, y_train_final, **fit_params)
    print("Treinamento concluído.")

    # 3. Fazer previsões e avaliar
    print("\nRealizando previsões no conjunto de teste...")
    y_pred_final = final_xgb_model.predict(X_test_scaled)
    print("Previsões concluídas.")

    # Recriar o LabelEncoder para mapear os nomes das classes
    label_encoder = LabelEncoder()
    label_encoder.fit(train_df['Classe'])

    accuracy_final = accuracy_score(y_test_encoded, y_pred_final)
    print(f'\nAcurácia do XGBoost final com a estratégia \'{strategy}\': {accuracy_final:.4f}')
    print('\nRelatório de Classificação:\n')
    print(classification_report(y_test_encoded, y_pred_final, target_names=label_encoder.classes_))


Usando os melhores hiperparâmetros da busca em todos os testes: {'subsample': 0.8, 'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.1, 'colsample_bytree': 1.0}

========================= INICIANDO TREINAMENTO COM A ESTRATÉGIA: 'SAMPLE_WEIGHT' =========================

Estratégia de balanceamento: Usando 'sample_weight'.

Iniciando o treinamento do modelo final...


/home/gabyl/projetos/trabalho-TEsClassification/venv/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [12:46:12] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Treinamento concluído.

Realizando previsões no conjunto de teste...
Previsões concluídas.

Acurácia do XGBoost final com a estratégia 'sample_weight': 0.3588

Relatório de Classificação:

              precision    recall  f1-score   support

    Helitron       0.16      0.39      0.23      7209
        LINE       0.05      0.19      0.08      3238
         LTR       0.79      0.36      0.49     56176
        MITE       0.16      0.47      0.24      7730
        SINE       0.07      0.20      0.10      1070
         TIR       0.42      0.34      0.38     24308

    accuracy                           0.36     99731
   macro avg       0.28      0.33      0.25     99731
weighted avg       0.57      0.36      0.41     99731


========================= INICIANDO TREINAMENTO COM A ESTRATÉGIA: 'SMOTE' =========================

Estratégia de balanceamento: Usando SMOTE. Isso pode levar alguns minutos...
SMOTE concluído.
  Shape original dos dados de treino: (367410, 64)
  Shape dos dados de 

/home/gabyl/projetos/trabalho-TEsClassification/venv/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [12:47:13] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Treinamento concluído.

Realizando previsões no conjunto de teste...
Previsões concluídas.

Acurácia do XGBoost final com a estratégia 'smote': 0.3658

Relatório de Classificação:

              precision    recall  f1-score   support

    Helitron       0.16      0.39      0.22      7209
        LINE       0.06      0.16      0.08      3238
         LTR       0.80      0.34      0.48     56176
        MITE       0.14      0.35      0.20      7730
        SINE       0.06      0.22      0.10      1070
         TIR       0.44      0.45      0.44     24308

    accuracy                           0.37     99731
   macro avg       0.27      0.32      0.25     99731
weighted avg       0.58      0.37      0.41     99731


========================= INICIANDO TREINAMENTO COM A ESTRATÉGIA: 'NONE' =========================

Nenhuma estratégia de balanceamento selecionada (treinamento padrão).

Iniciando o treinamento do modelo final...


/home/gabyl/projetos/trabalho-TEsClassification/venv/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [12:47:41] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Treinamento concluído.

Realizando previsões no conjunto de teste...
Previsões concluídas.

Acurácia do XGBoost final com a estratégia 'none': 0.5891

Relatório de Classificação:

              precision    recall  f1-score   support

    Helitron       0.92      0.00      0.00      7209
        LINE       0.65      0.01      0.01      3238
         LTR       0.78      0.69      0.73     56176
        MITE       0.11      0.00      0.00      7730
        SINE       0.74      0.06      0.10      1070
         TIR       0.40      0.81      0.54     24308

    accuracy                           0.59     99731
   macro avg       0.60      0.26      0.23     99731
weighted avg       0.64      0.59      0.55     99731

